In [30]:
import pyarrow as pa
import pyarrow.parquet as pq
import os

https://towardsdatascience.com/parquet-best-practices-discover-your-data-without-loading-them-f854c57a45b6

In [2]:
def get_first_parquet_from_path(path):
    for (dir_path, _, files) in os.walk(path):
        for f in files:
            if f.endswith(".parquet"):
                first_pq_path = os.path.join(dir_path, f)
                return first_pq_path

In [7]:
path = 'APPLICATIONS_PROCESSED'
first_pq = get_first_parquet_from_path(path)
print(first_pq)
#Output : APPLICATIONS_PARTITIONED/NAME_INCOME_TYPE=Commercial associate/CODE_GENDER=F/6183f182ab0b47c49cf56a3e09a3a7b1-0.parquet

APPLICATIONS_PROCESSED/NAME_INCOME_TYPE=Student/fcfd451199bf42b1989ceca04b7209e5-0.parquet


In [14]:
%%time
first_ds = pq.read_table(first_pq)
first_ds.num_rows, first_ds.num_columns, first_ds.schema

CPU times: user 66.7 ms, sys: 30.6 ms, total: 97.3 ms
Wall time: 13.9 ms


(170,
 18,
 ID: int64
 CODE_GENDER: string
 FLAG_OWN_CAR: string
 FLAG_OWN_REALTY: string
 CNT_CHILDREN: int64
 AMT_INCOME_TOTAL: double
 NAME_EDUCATION_TYPE: string
 NAME_FAMILY_STATUS: string
 NAME_HOUSING_TYPE: string
 DAYS_EMPLOYED: int64
 FLAG_MOBIL: bool
 FLAG_WORK_PHONE: bool
 FLAG_PHONE: bool
 FLAG_EMAIL: bool
 OCCUPATION_TYPE: string
 CNT_FAM_MEMBERS: double
 MONTH_INCOME_TOTAL: double
 AGE: double
 -- schema metadata --
 pandas: '{"index_columns": [], "column_indexes": [{"name": null, "field_n' + 2567)

In [15]:
%%time
ts=pq.read_metadata(first_pq)
ts.num_rows, ts.num_columns, pq.read_schema(first_pq)

CPU times: user 2.36 ms, sys: 994 µs, total: 3.35 ms
Wall time: 2.47 ms


(170,
 18,
 ID: int64
 CODE_GENDER: string
 FLAG_OWN_CAR: string
 FLAG_OWN_REALTY: string
 CNT_CHILDREN: int64
 AMT_INCOME_TOTAL: double
 NAME_EDUCATION_TYPE: string
 NAME_FAMILY_STATUS: string
 NAME_HOUSING_TYPE: string
 DAYS_EMPLOYED: int64
 FLAG_MOBIL: bool
 FLAG_WORK_PHONE: bool
 FLAG_PHONE: bool
 FLAG_EMAIL: bool
 OCCUPATION_TYPE: string
 CNT_FAM_MEMBERS: double
 MONTH_INCOME_TOTAL: double
 AGE: double
 -- schema metadata --
 pandas: '{"index_columns": [], "column_indexes": [{"name": null, "field_n' + 2567)

In [16]:
%%time
parquet_file = pq.ParquetFile(first_pq)
ts=parquet_file.metadata.row_group(0)
for nm in range(ts.num_columns):
    print(ts.column(nm))

  file_offset: 165
  file_path: 
  physical_type: INT64
  num_values: 13
  path_in_schema: ID
  is_stats_set: True
  statistics:
      has_min_max: True
      min: 1464
      max: 16564
      null_count: 0
      distinct_count: None
      num_values: 13
      physical_type: INT64
      logical_type: None
      converted_type (legacy): NONE
  compression: SNAPPY
  encodings: ('PLAIN', 'RLE', 'RLE_DICTIONARY')
  has_dictionary_page: True
  dictionary_page_offset: 4
  data_page_offset: 86
  total_compressed_size: 161
  total_uncompressed_size: 197
  file_offset: 324
  file_path: 
  physical_type: BYTE_ARRAY
  num_values: 13
  path_in_schema: CODE_GENDER
  is_stats_set: True
  statistics:
      has_min_max: True
      min: F
      max: M
      null_count: 0
      distinct_count: None
      num_values: 13
      physical_type: BYTE_ARRAY
      logical_type: String
      converted_type (legacy): UTF8
  compression: SNAPPY
  encodings: ('PLAIN', 'RLE', 'RLE_DICTIONARY')
  has_dictionary_page: 

In [23]:
import pandas as pd
beautiful_df = pd.DataFrame()
for nm in range(ts.num_columns):
    path_in_schema = ts.column(nm).path_in_schema
    compressed_size = ts.column(nm).total_compressed_size
    stats = ts.column(nm).statistics
    min_value = stats.min
    max_value = stats.max
    physical_type = stats.physical_type
    beautiful_df[path_in_schema] = pd.DataFrame([physical_type, min_value, max_value, compressed_size])
df = beautiful_df.T
df.columns = ['DTYPE', 'Min', 'Max', 'Compressed_Size_(KO)']
print(df)

                          DTYPE                Min  \
ID                        INT64               1464   
CODE_GENDER          BYTE_ARRAY                  F   
FLAG_OWN_CAR         BYTE_ARRAY                  N   
FLAG_OWN_REALTY      BYTE_ARRAY                  N   
CNT_CHILDREN              INT64                  0   
AMT_INCOME_TOTAL         DOUBLE           112500.0   
NAME_EDUCATION_TYPE  BYTE_ARRAY   Higher education   
NAME_FAMILY_STATUS   BYTE_ARRAY     Civil marriage   
NAME_HOUSING_TYPE    BYTE_ARRAY  House / apartment   
DAYS_EMPLOYED             INT64              -3904   
FLAG_MOBIL              BOOLEAN               True   
FLAG_WORK_PHONE         BOOLEAN              False   
FLAG_PHONE              BOOLEAN              False   
FLAG_EMAIL              BOOLEAN              False   
OCCUPATION_TYPE      BYTE_ARRAY         Core staff   
CNT_FAM_MEMBERS          DOUBLE                1.0   
MONTH_INCOME_TOTAL       DOUBLE             9375.0   
AGE                      DOU

In [24]:
def get_all_partitions(path):
    partitions = {}
    i = 0
    for (_, partitions_layer, _) in os.walk(path):
        if len(partitions_layer)>0:
            key = partitions_layer[0].split('=')[0]
            partitions[key] = sorted([partitions_layer[i].split('=')[1] for i in range(len(partitions_layer))])
        else:
            break
    return partitions

In [25]:
ps = get_all_partitions(path)
ps.keys(), ps.values()

(dict_keys(['NAME_INCOME_TYPE']),
 dict_values([['Commercial%20associate', 'Pensioner', 'State%20servant', 'Student', 'Working']]))

In [27]:
df_pensioner = pd.read_parquet('APPLICATIONS_PROCESSED/NAME_INCOME_TYPE=Pensioner/')

From the official documentation 512MB — 1GB is the optimal size for a partition

In [28]:
pq_table = pq.read_table('APPLICATIONS_PROCESSED')
pq.write_to_dataset(pq_table, 'APPLICATIONS_REPARTITIONED', partition_cols=['NAME_INCOME_TYPE'])

In [29]:
partitions = get_all_partitions('APPLICATIONS_REPARTITIONED')
partitions.keys(), partitions.values()

(dict_keys(['NAME_INCOME_TYPE']),
 dict_values([['Commercial%20associate', 'Pensioner', 'State%20servant', 'Student', 'Working']]))